# Body Composition Analysis
### Random Forest Regression with Exercise Support

This notebook implements **Random Forest Regressor** models to predict body composition changes across multiple time horizons (3, 6, 9, and 12 months). It forecasts:

- 📉 **Body Fat Percentage (BFP)** changes
- 💪 **Muscle Mass** changes
- 🏆 **Definition Score** (1–5 scale)

Exercise data (exercise name, target muscle group, category) is incorporated as features alongside standard physiological and nutritional variables.

---
## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict
import joblib

---
## 2. Data Loading

Load the fat loss dataset from `fatLoss.csv` and display a quick overview.

In [2]:
try:
    df = pd.read_csv('fatLoss.csv')
    print('Dataset loaded successfully!')
except:
    print('Error loading dataset')
    exit()

print('Dataset Overview:')
print(df.head())
print('\nDataset Info:')
print(df.info())

Dataset loaded successfully!
Dataset Overview:
         exercise_name target_muscle_group      type  age gender  sets  reps  \
0  Barbell Bench Press               Chest  Compound   32      M     4    10   
1  Barbell Bench Press               Chest  Compound   28      F     3    12   
2  Barbell Bench Press               Chest  Compound   39      M     5     8   
3  Barbell Bench Press               Chest  Compound   35      F     4    10   
4  Barbell Bench Press               Chest  Compound   26      M     3    12   

   weight  frequency  protein  calories  sleep    experience  \
0      80          3      150      2800    7.5  Intermediate   
1      45          2      130      2600    7.2      Beginner   
2      90          4      170      3100    8.0      Advanced   
3      50          3      135      2700    7.4  Intermediate   
4      85          2      145      2750    7.6  Intermediate   

   genetic_advantage  actual_fat_loss  daily_deficit  current_weight  height  
0       

---
## 3. Data Preparation

### 3.1 BMR Calculation

Compute **Basal Metabolic Rate (BMR)** using the Mifflin–St Jeor equation, applied separately for males and females.

| Gender | Formula |
|--------|---------|
| Male   | `88.362 + (13.397 × weight) + (4.799 × height) − (5.677 × age)` |
| Female | `447.593 + (9.247 × weight) + (3.098 × height) − (4.330 × age)` |

In [3]:
# BMR for Males
df.loc[df['gender'] == 'M', 'BMR'] = (
    88.362
    + (13.397 * df.loc[df['gender'] == 'M', 'current_weight'])
    + (4.799  * df.loc[df['gender'] == 'M', 'height'])
    - (5.677  * df.loc[df['gender'] == 'M', 'age'])
)

# BMR for Females
df.loc[df['gender'] == 'F', 'BMR'] = (
    447.593
    + (9.247 * df.loc[df['gender'] == 'F', 'current_weight'])
    + (3.098 * df.loc[df['gender'] == 'F', 'height'])
    - (4.330 * df.loc[df['gender'] == 'F', 'age'])
)

### 3.2 Initial BFP & Exercise Data

Generate an initial **Body Fat Percentage (BFP)** column and simulate exercise-related columns if they are not already present in the dataset.

In [4]:
# Generate initial BFP
np.random.seed(42)
df['BFP'] = np.random.uniform(5, 50, size=len(df))

# Simulate exercise_name if not present
if 'exercise_name' not in df.columns:
    exercise_names = [
        'Barbell Bench Press', 'Barbell Rows', 'Squats', 'Deadlifts',
        'Pull-ups', 'Dumbbell Press', 'Lat Pulldowns', 'Bicep Curls',
        'Tricep Extensions', 'Leg Press'
    ]
    df['exercise_name'] = np.random.choice(exercise_names, size=len(df))

# Simulate target_muscle_group if not present
if 'target_muscle_group' not in df.columns:
    muscle_groups = ['Chest', 'Back', 'Legs', 'Shoulders', 'Arms']
    df['target_muscle_group'] = np.random.choice(muscle_groups, size=len(df))

# Derive exercise_category from exercise name
if 'exercise_category' not in df.columns:
    compound_exercises = [
        'Barbell Bench Press', 'Barbell Rows', 'Squats',
        'Deadlifts', 'Pull-ups', 'Lat Pulldowns'
    ]
    df['exercise_category'] = df['exercise_name'].apply(
        lambda x: 'Compound' if x in compound_exercises else 'Isolation'
    )

print('BMR, BFP, and exercise data prepared!')

BMR, BFP, and exercise data prepared!


---
## 4. Feature Engineering

Derive additional body composition features from the raw data:

| Feature | Description |
|---------|-------------|
| `BMI` | Body Mass Index |
| `activity_level` | Frequency × 5 |
| `TDEE` | Total Daily Energy Expenditure |
| `estimated_fat_mass` | BFP% × current weight |
| `muscle_mass` | Weight − fat mass |
| `training_intensity` | Volume normalised by body weight |
| `sleep_quality` | Sleep score (1.0 if ≥ 7.5 hrs) |
| `protein_per_kg` | Protein intake per kg of body weight |
| `deficit_ratio` | Daily deficit / TDEE |
| `volume` | Sets × reps × weight |
| `intensity` | Weight per rep |
| `calories_per_kg` | Calories per kg of body weight |

In [5]:
def calculate_body_composition_features(df):
    """Calculate additional features for body composition prediction."""

    df['BMI']             = df['current_weight'] / ((df['height'] / 100) ** 2)
    df['activity_level']  = df['frequency'] * 5
    df['TDEE']            = df['BMR'] * (1.2 + (df['activity_level'] * 0.1))

    df['estimated_fat_mass'] = (df['BFP'] / 100) * df['current_weight']
    df['muscle_mass']        = df['current_weight'] - df['estimated_fat_mass']

    df['training_intensity'] = (df['weight'] * df['sets'] * df['reps']) / df['current_weight']
    df['sleep_quality']      = np.where(df['sleep'] >= 7.5, 1, df['sleep'] / 7.5)
    df['protein_per_kg']     = df['protein'] / df['current_weight']
    df['deficit_ratio']      = df['daily_deficit'] / df['TDEE']

    df['volume']         = df['sets'] * df['reps'] * df['weight']
    df['intensity']      = df['weight'] / np.maximum(df['reps'], 1)
    df['calories_per_kg'] = df['calories'] / np.maximum(df['current_weight'], 1)

    return df


df = calculate_body_composition_features(df)
print('Feature engineering completed!')

Feature engineering completed!


---
## 5. Categorical Encoding

Use `LabelEncoder` to convert categorical columns into numeric representations suitable for tree-based models.

In [6]:
le_gender            = LabelEncoder()
le_experience        = LabelEncoder()
le_type              = LabelEncoder()
le_exercise_name     = LabelEncoder()
le_muscle_group      = LabelEncoder()
le_exercise_category = LabelEncoder()

df['gender_encoded']            = le_gender.fit_transform(df['gender'])
df['experience_encoded']        = le_experience.fit_transform(df['experience'])
df['type_encoded']              = le_type.fit_transform(df['type'])
df['exercise_name_encoded']     = le_exercise_name.fit_transform(df['exercise_name'])
df['muscle_group_encoded']      = le_muscle_group.fit_transform(df['target_muscle_group'])
df['exercise_category_encoded'] = le_exercise_category.fit_transform(df['exercise_category'])

---
## 6. Target Variable Generation

Simulate ground-truth labels for **BFP**, **muscle mass**, and **definition score** at 3, 6, 9, and 12 months.

- BFP decreases progressively; floored at physiological minimums (3% M / 8% F).
- Muscle mass grows proportionally to training intensity.
- Definition score is derived from the muscle-mass-to-BFP ratio and scaled to **1–5**.

In [7]:
np.random.seed(42)

df['predicted_weight_after_loss'] = df['current_weight'] - df['actual_fat_loss']

# BFP projections — diminishing returns over time
df['bfp_at_3']  = df['BFP'] - np.random.uniform(2.0, 5.0, size=len(df))
df['bfp_at_6']  = df['bfp_at_3']  - np.random.uniform(1.0, 3.0, size=len(df))
df['bfp_at_9']  = df['bfp_at_6']  - np.random.uniform(0.5, 2.0, size=len(df))
df['bfp_at_12'] = df['bfp_at_9']  - np.random.uniform(0.5, 1.5, size=len(df))

# Clamp to physiological minimums
min_bfp = np.where(df['gender'] == 'M', 3, 8)
for col in ['bfp_at_3', 'bfp_at_6', 'bfp_at_9', 'bfp_at_12']:
    df[col] = np.maximum(df[col], min_bfp)

# Muscle mass projections
muscle_gain_factor = 1 + (df['training_intensity'] * 0.001)
for months, bfp_col, mm_col in [
    (3,  'bfp_at_3',  'muscle_mass_at_3'),
    (6,  'bfp_at_6',  'muscle_mass_at_6'),
    (9,  'bfp_at_9',  'muscle_mass_at_9'),
    (12, 'bfp_at_12', 'muscle_mass_at_12'),
]:
    df[mm_col] = (
        df['predicted_weight_after_loss'] * (1 - df[bfp_col] / 100)
    ) * muscle_gain_factor

# Definition score (1–5)
def calculate_definition(muscle_mass, bfp):
    """Higher muscle mass + lower BFP → higher definition."""
    definition_raw = (muscle_mass / 10) * (50 - bfp) / 10
    return np.clip(definition_raw / definition_raw.max() * 4 + 1, 1, 5)

df['definition_at_3']  = calculate_definition(df['muscle_mass_at_3'],  df['bfp_at_3'])
df['definition_at_6']  = calculate_definition(df['muscle_mass_at_6'],  df['bfp_at_6'])
df['definition_at_9']  = calculate_definition(df['muscle_mass_at_9'],  df['bfp_at_9'])
df['definition_at_12'] = calculate_definition(df['muscle_mass_at_12'], df['bfp_at_12'])

print('Target variables generated for all time intervals!')
print(f'BFP range at 12 months: {df["bfp_at_12"].min():.1f}% – {df["bfp_at_12"].max():.1f}%')
print(f'Definition score range at 12 months: {df["definition_at_12"].min():.1f} – {df["definition_at_12"].max():.1f}')

Target variables generated for all time intervals!
BFP range at 12 months: 3.0% – 41.1%
Definition score range at 12 months: 1.4 – 5.0


---
## 7. Model Training

Train a **Random Forest Regressor** (`n_estimators=100`, `max_depth=10`) for each combination of time interval × metric — **12 models** in total.

Performance is evaluated using **MSE** and **R²** on a 20% hold-out test set.

In [8]:
features = [
    'age', 'gender_encoded', 'current_weight', 'height', 'BMI',
    'sets', 'reps', 'weight', 'frequency', 'training_intensity',
    'protein', 'protein_per_kg', 'calories', 'sleep', 'sleep_quality',
    'experience_encoded', 'genetic_advantage', 'daily_deficit', 'deficit_ratio',
    'type_encoded', 'BFP', 'muscle_mass', 'TDEE', 'activity_level',
    'exercise_name_encoded', 'muscle_group_encoded', 'exercise_category_encoded',
    'volume', 'intensity', 'calories_per_kg'
]

missing_features = [f for f in features if f not in df.columns]
if missing_features:
    print(f'Missing features: {missing_features}')
else:
    X = df[features]
    print(f'Feature matrix prepared with {len(features)} features')
    print(f'Dataset shape: {X.shape}')

Feature matrix prepared with 30 features
Dataset shape: (390, 30)


In [9]:
time_intervals = [3, 6, 9, 12]
metrics        = ['bfp', 'muscle_mass', 'definition']

models       = {}
model_scores = {}

for interval in time_intervals:
    models[interval]       = {}
    model_scores[interval] = {}

    for metric in metrics:
        target_col = f'{metric}_at_{interval}'
        y = df[target_col]

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )

        rf = RandomForestRegressor(n_estimators=100, random_state=42, max_depth=10)
        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_test)
        mse    = mean_squared_error(y_test, y_pred)
        r2     = r2_score(y_test, y_pred)

        models[interval][metric]       = rf
        model_scores[interval][metric] = {'mse': mse, 'r2': r2}

        print(f'{metric.upper()} at {interval} months — MSE: {mse:.3f}, R²: {r2:.3f}')

print('\nAll models trained successfully!')

BFP at 3 months — MSE: 0.049, R²: 1.000
MUSCLE_MASS at 3 months — MSE: 1.603, R²: 0.984
DEFINITION at 3 months — MSE: 0.007, R²: 0.992
BFP at 6 months — MSE: 0.526, R²: 0.996
MUSCLE_MASS at 6 months — MSE: 2.061, R²: 0.979
DEFINITION at 6 months — MSE: 0.011, R²: 0.987
BFP at 9 months — MSE: 0.615, R²: 0.995
MUSCLE_MASS at 9 months — MSE: 2.231, R²: 0.977
DEFINITION at 9 months — MSE: 0.017, R²: 0.980
BFP at 12 months — MSE: 0.674, R²: 0.995
MUSCLE_MASS at 12 months — MSE: 2.262, R²: 0.977
DEFINITION at 12 months — MSE: 0.021, R²: 0.975

All models trained successfully!


---
## 8. Prediction

### 8.1 Helper: Safe Label Encoding

Gracefully handles unseen categories at inference time by falling back to a default encoded value.

In [10]:
def safe_transform(encoder, value, default_value=0):
    """Safely transform a value using a label encoder.
    Returns default_value if the label was not seen during training.
    """
    try:
        if value in encoder.classes_:
            return encoder.transform([value])[0]
        else:
            print(f"Warning: '{value}' not in training data — using default")
            return default_value
    except Exception:
        return default_value

### 8.2 Multi-Exercise Body Composition Predictor

Accepts a list of exercises (each with name, sets, reps, weight, muscle group, and category) and returns projected **BFP**, **muscle mass**, and **definition score** at 3, 6, 9, and 12 months.

In [11]:
def predict_body_composition_journey_with_exercises(
    age: int,
    gender: str,
    exercises: List[Dict],
    frequency: int,
    protein: float,
    calories: int,
    sleep: float,
    experience: str,
    genetic_advantage: int = 3,
    daily_deficit: float = 500,
    initial_bfp: float = 20.0,
    current_weight: float = 80.0,
    height: float = 180.0,
):
    """
    Predict the complete body composition journey for a new user.

    Parameters
    ----------
    age             : User age (years)
    gender          : 'M' or 'F'
    exercises       : List of exercise dicts with keys:
                      exercise_name, sets, reps, weight,
                      target_muscle_group, exercise_category
    frequency       : Training days per week
    protein         : Daily protein intake (g)
    calories        : Daily calorie intake
    sleep           : Average nightly sleep (hours)
    experience      : Training experience level
    genetic_advantage : Genetic score 1–5 (default 3)
    daily_deficit   : Daily caloric deficit (kcal)
    initial_bfp     : Starting body fat percentage
    current_weight  : Current body weight (kg)
    height          : Height (cm)

    Returns
    -------
    dict  : {interval: {metric: value}} for bfp, muscle_mass, definition
    """
    try:
        difficulty_weights = {'Compound': 1.0, 'Isolation': 0.8}

        # Identify primary exercise by highest weighted volume
        total_volume     = 0
        primary_exercise = None
        max_volume       = 0

        for ex in exercises:
            vol = (
                ex['sets'] * ex['reps'] * ex['weight']
                * difficulty_weights.get(ex.get('exercise_category', 'Compound'), 1.0)
            )
            total_volume += vol
            if vol > max_volume:
                max_volume       = vol
                primary_exercise = ex

        # Equivalent single-exercise representation
        equiv_sets   = min(10, primary_exercise['sets'])
        equiv_reps   = min(20, primary_exercise['reps'])
        equiv_weight = min(300, total_volume / (equiv_sets * equiv_reps))

        # BMR
        if gender == 'M':
            bmr = 88.362 + (13.397 * current_weight) + (4.799 * height) - (5.677 * age)
        else:
            bmr = 447.593 + (9.247 * current_weight) + (3.098 * height) - (4.330 * age)

        # Derived features
        bmi              = current_weight / ((height / 100) ** 2)
        activity_level   = frequency * 5
        tdee             = bmr * (1.2 + (activity_level * 0.1))
        estimated_fat    = (initial_bfp / 100) * current_weight
        muscle_mass      = current_weight - estimated_fat
        training_int     = (equiv_weight * equiv_sets * equiv_reps) / current_weight
        sleep_quality    = 1 if sleep >= 7.5 else sleep / 7.5
        protein_per_kg   = protein / current_weight
        deficit_ratio    = daily_deficit / tdee
        volume           = equiv_sets * equiv_reps * equiv_weight
        intensity        = equiv_weight / max(equiv_reps, 1)
        calories_per_kg  = calories / max(current_weight, 1)

        # Encode categoricals
        gender_enc    = safe_transform(le_gender,            gender)
        exp_enc       = safe_transform(le_experience,        experience)
        ex_name_enc   = safe_transform(le_exercise_name,     primary_exercise['exercise_name'])
        mg_enc        = safe_transform(le_muscle_group,      primary_exercise.get('target_muscle_group', 'Chest'))
        ex_cat_enc    = safe_transform(le_exercise_category, primary_exercise.get('exercise_category',   'Compound'))
        type_enc      = 0  # Not provided via exercise input

        user_data = np.array([[
            age, gender_enc, current_weight, height, bmi,
            equiv_sets, equiv_reps, equiv_weight, frequency, training_int,
            protein, protein_per_kg, calories, sleep, sleep_quality,
            exp_enc, genetic_advantage, daily_deficit, deficit_ratio,
            type_enc, initial_bfp, muscle_mass, tdee, activity_level,
            ex_name_enc, mg_enc, ex_cat_enc,
            volume, intensity, calories_per_kg
        ]])

        predictions = {}
        for interval in time_intervals:
            predictions[interval] = {}
            for metric in metrics:
                pred = models[interval][metric].predict(user_data)[0]
                predictions[interval][metric] = max(pred, 0)

        return predictions

    except Exception as e:
        print(f'Error in prediction: {e}')
        return None

---
## 9. Example Prediction

Run the predictor for a 28-year-old male performing a push/pull/legs programme at a 500 kcal/day deficit.

In [12]:
exercises = [
    {
        'exercise_name': 'Barbell Bench Press',
        'sets': 4, 'reps': 10, 'weight': 80,
        'target_muscle_group': 'Chest',
        'exercise_category': 'Compound',
    },
    {
        'exercise_name': 'Barbell Rows',
        'sets': 4, 'reps': 10, 'weight': 75,
        'target_muscle_group': 'Back',
        'exercise_category': 'Compound',
    },
    {
        'exercise_name': 'Squats',
        'sets': 4, 'reps': 12, 'weight': 100,
        'target_muscle_group': 'Legs',
        'exercise_category': 'Compound',
    },
]

journey = predict_body_composition_journey_with_exercises(
    age=28,
    gender='M',
    exercises=exercises,
    frequency=4,
    protein=150,
    calories=2200,
    sleep=8.0,
    experience='Intermediate',
    genetic_advantage=4,
    daily_deficit=500,
    initial_bfp=18.0,
    current_weight=85.0,
    height=180.0,
)

c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
c:\Python312\Lib\site-packages\sklearn\utils\validation.py:2

In [13]:
if journey:
    print('=' * 55)
    print('   BODY COMPOSITION JOURNEY PREDICTION WITH EXERCISES')
    print('=' * 55)
    print(f'Starting Stats : Weight 85 kg | BFP 18.0%')
    print(f'Programme      : {len(exercises)} compound exercises')
    print('-' * 55)

    for interval in time_intervals:
        print(f'\nMonth {interval:>2}:')
        print(f'  Body Fat      : {journey[interval]["bfp"]:.1f}%')
        print(f'  Muscle Mass   : {journey[interval]["muscle_mass"]:.1f} kg')
        print(f'  Definition    : {journey[interval]["definition"]:.1f} / 5')

   BODY COMPOSITION JOURNEY PREDICTION WITH EXERCISES
Starting Stats : Weight 85 kg | BFP 18.0%
Programme      : 3 compound exercises
-------------------------------------------------------

Month  3:
  Body Fat      : 15.2%
  Muscle Mass   : 61.0 kg
  Definition    : 3.4 / 5

Month  6:
  Body Fat      : 13.5%
  Muscle Mass   : 62.5 kg
  Definition    : 3.6 / 5

Month  9:
  Body Fat      : 12.0%
  Muscle Mass   : 63.7 kg
  Definition    : 3.8 / 5

Month 12:
  Body Fat      : 11.1%
  Muscle Mass   : 64.2 kg
  Definition    : 3.8 / 5


---
## 10. Save Models & Encoders

Persist all trained models and label encoders to disk using `joblib` for downstream inference or deployment.

In [14]:
import os
os.makedirs('models', exist_ok=True)

joblib.dump(models,               'models/body_composition_models.pkl')
joblib.dump(le_gender,            'models/le_gender_body.pkl')
joblib.dump(le_experience,        'models/le_experience_body.pkl')
joblib.dump(le_exercise_name,     'models/le_exercise_name_body.pkl')
joblib.dump(le_muscle_group,      'models/le_muscle_group_body.pkl')
joblib.dump(le_exercise_category, 'models/le_exercise_category_body.pkl')

print('Models and encoders saved successfully!')
print(f'Available exercises   : {len(le_exercise_name.classes_)}')
print(f'Available muscle groups: {len(le_muscle_group.classes_)}')

Models and encoders saved successfully!
Available exercises   : 39
Available muscle groups: 4
